<a href="https://colab.research.google.com/github/vannsoko/MNIST/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

In [7]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL Image to tensor (C x H x W)
])

# Load the MNIST dataset
dataset = datasets.MNIST(
    root='./data',          # Directory to store the dataset
    train=True,             # Load the training set
    download=True,          # Download if not available
    transform=transform,    # Apply the transform
)
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform,
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)



100%|██████████| 9.91M/9.91M [00:00<00:00, 137MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 9.77MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 118MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.69MB/s]


lr=0.001 => loss=1.4612 ep=7 : 97.6%

[I 2025-10-07 19:00:52,894] Trial 49 finished with value: 1.4865225198421073 and parameters: {'lr': 0.0002896285253050259, 'batch_size': 16, 'hidden_size1': 641, 'hidden_size2': 403, 'dropout': 0.21759572322363618}. Best is trial 31 with value: 1.4832874181422782.
Test Loss: 1.481069841980934; Test accuracy: 97.94
Best params: {'lr': 0.0002154130042128118, 'batch_size': 32, 'hidden_size1': 822, 'hidden_size2': 804, 'dropout': 0.11041526551552969}
Best value: 1.4832874181422782

In [13]:
class MNIST(nn.Module):
  def __init__(self, h1, h2, dropout):
    super(MNIST, self).__init__()
    self.flatten = nn.Flatten()
    self.linear_rellu_stack = nn.Sequential(
        nn.Linear(28*28, h1),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(h1, h2),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(h2, 10),
    )
    self.softmax = nn.Softmax(dim=1)


  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_rellu_stack(x)
    return self.softmax(logits)

In [21]:
model = MNIST(h1=256, h2=128, dropout=0.1).to(device)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
for epoch in tqdm(range(20)):
  for images, labels in train_loader:
    # Move tensors to the configured device
    images = images.to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Backward pass and optimisation
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
  print(f' Loss: {loss.item():.4f}')

  5%|▌         | 1/20 [00:07<02:21,  7.45s/it]

 Loss: 1.5700


 10%|█         | 2/20 [00:15<02:22,  7.92s/it]

 Loss: 1.5382


 15%|█▌        | 3/20 [00:23<02:12,  7.81s/it]

...

 Loss: 1.4673


 95%|█████████▌| 19/20 [02:32<00:08,  8.31s/it]

 Loss: 1.4693


100%|██████████| 20/20 [02:39<00:00,  7.97s/it]

 Loss: 1.4714


In [22]:
counter = 0
correct = 0
model.eval()
with torch.no_grad():
  for image, true_label in test_dataset:
    image = image.unsqueeze(0).to(device)
    output = model(image)
    predicted_label = torch.argmax(output, dim=1).item()
    if true_label==predicted_label:
      correct += 1
    counter += 1

print(f"{correct=}, {counter=}")
print(f"{100*correct/counter}%")


correct=9777, counter=10000
97.77%


For Optimization:

In [ ]:
pip install --upgrade optuna

In [5]:
import optuna

In [ ]:
def train_epoch(model, optimizer, batch_size):
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  criterion = nn.CrossEntropyLoss()
  model.train()
  for images, labels in train_loader:
    # Move tensors to the configured device
    images = images.to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Backward pass and optimisation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print(f"Loss: {loss.item():.4f}")
  return loss.item()

def validate(model, dataset, batch_size):
  model.eval()
  criterion = nn.CrossEntropyLoss()
  total_loss = 0
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in dataset:
      # Move tensors to the configured device
      images = images.to(device)
      labels = labels.to(device)

      # Forward pass
      outputs = model(images)
      loss = criterion(outputs, labels)
      total_loss += loss.item()

      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  avg_loss = total_loss / len(dataset)
  accuracy = 100 * correct / total
  return avg_loss, accuracy

def objective(trial):
    # Suggest hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
    hidden_size1 = trial.suggest_int('hidden_size1', 256, 2056)
    hidden_size2 = trial.suggest_int('hidden_size2', 128, 2056)

    dropout = trial.suggest_float('dropout', 0.05, 0.25)
    #n_layers = trial.suggest_int('n_layers', 1, 4)

    # Build model
    model = MNIST(h1=hidden_size1,
                      h2=hidden_size2,
                      dropout=dropout,
                      ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Train
    for epoch in tqdm(range(40)):
        train_loss = train_epoch(model, optimizer, batch_size)
        val_loss, val_accuracy = validate(model, val_loader, batch_size)
        test_loss, test_accuracy = validate(model, test_loader, batch_size)

        print(f"Test Loss: {test_loss}; Test accuracy: {test_accuracy}")
        trial.set_user_attr('test_loss', test_loss)
        trial.set_user_attr('test_accuracy', test_accuracy)
        trial.set_user_attr('final_train_loss', train_loss)
        # Report intermediate value for pruning
        trial.report(val_loss, epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()



    return test_accuracy

In [ ]:
# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Best params: {study.best_params}")
print(f"Best value: {study.best_value}")